# Full-Mamba Source-Only (Hoisted Scan) — Zhou2016 LOSO
Enable **GPU** and **Internet** in Kaggle, then run the cell below. Each MI trial is filtered at 8–30 Hz and uses the interval `[0.5, 5.0 s)`. Primary protocol: source Sessions 1–2 train, source Session 3 validates checkpoints, target Sessions 1–2 never enter training or adaptation, and target Session 3 is the held-out test set.

The Mamba blocks use the hoisted selective scan (`mamba_scan_mode: hoisted`, pure PyTorch, no custom CUDA kernel): bit-identical to the scripted scan, with only one multiply-add per time step inside the loop. Before training, `scripts/benchmark_scan_modes.py` prints latency for TCFormer vs Mamba scripted/parallel/hoisted on CPU and GPU.

In [ ]:
# One-cell Kaggle: Zhou2016 LOSO targets S01-S04. Enable GPU + Internet.
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/full-mamba-source-only-parallel-scan'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/tcformer-full-mamba-parallel-scan-zhou2016')
SUBJECT_IDS = list(range(1, 5))
MAX_EPOCHS = 100
BATCH_SIZE = 32
USE_RA = False
RUN_TAG = 'source-only-no-ra-hoisted-scan'
MNE_DATA = Path('/kaggle/working/mne_data')
RESULT_ARCHIVE = Path(f'/kaggle/working/full_mamba_source_only_zhou2016_loso_mi8_30_t0p5_5p0_ep{MAX_EPOCHS}_{RUN_TAG}_results')
TRAIN_LOG = Path(f'/kaggle/working/full_mamba_source_only_zhou2016_loso_mi8_30_t0p5_5p0_ep{MAX_EPOCHS}_{RUN_TAG}.log')

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                heartbeat = f'[Kaggle heartbeat] process is running | elapsed={(time.monotonic() - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    # A previous run modifies the YAML config locally. Force the disposable
    # Kaggle clone back to the fetched branch so rerunning this cell is safe.
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
pipeline_text = (REPO_PATH / 'train_pipeline.py').read_text(encoding='utf-8')
config_text = (REPO_PATH / 'configs/full_mamba_source_only.yaml').read_text(encoding='utf-8')
assert 'ModelCheckpoint' in pipeline_text and 'best_model_path' in pipeline_text
assert 'Pure source-only evaluation' in pipeline_text
assert 'domain_adaptation: false' in config_text and 'riemannian_alignment: false' in config_text
assert 'model: "FullMambaSourceOnly"' in config_text
assert 'mamba_scan_mode: hoisted' in config_text
assert '_hoisted_selective_scan' in (REPO_PATH / 'models/tcformer.py').read_text(encoding='utf-8')
print('Verified: pure source-only graph, hoisted Mamba scan and best source-validation checkpoint selection.')

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

override = f"import yaml; from pathlib import Path; p=Path('configs/full_mamba_source_only.yaml'); c=yaml.safe_load(p.read_text()); c['subject_ids']={SUBJECT_IDS!r}; c['max_epochs_loso_zhou2016']={MAX_EPOCHS}; pre=c['preprocessing']['zhou2016']; pre['low_cut']=8.0; pre['high_cut']=30.0; pre['start']=0.5; pre['trial_duration']=4.5; pre['batch_size']={BATCH_SIZE}; pre['riemannian_alignment']={USE_RA!r}; pre['interaug']=False; pre.setdefault('model_overrides', {{}}).update({{'lr': 0.001, 'weight_decay': 0.0001}}); p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])

# Latency check before training: TCFormer vs Mamba scripted/parallel/hoisted
# (CPU and GPU, batch 1 and 32) plus numerical equivalence to the scripted scan.
run([PYTHON, 'scripts/benchmark_scan_modes.py', '--channels', '14', '--samples', '1125', '--classes', '3'], cwd=REPO_PATH, env={**os.environ, 'MPLBACKEND': 'Agg'})

MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_ZHOU2016_PATH': str(MNE_DATA)})
print('===== FULL-MAMBA SOURCE-ONLY (HOISTED SCAN) | ZHOU2016 LOSO S01-S04 | 3 CLASSES | 8--30 Hz | 0.5--5.0 s | 1125 SAMPLES =====', flush=True)
print(f'Options: epochs={MAX_EPOCHS} | batch={BATCH_SIZE} | lr=0.001 | weight_decay=0.0001 | Filter=8--30 Hz | RA=False | DA=False | IM-TTA=False | InterAug=False | MambaScan=hoisted', flush=True)
print('Primary protocol: source S1-S2=train, source S3=validation, target S1-S2=unused in training, target S3=held-out test; all target sessions=auxiliary test only', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'full_mamba_source_only', '--dataset', 'zhou2016', '--loso', '--gpu_id', '0', '--no_interaug'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Zhou2016 LOSO complete. Kaggle output:', archive, flush=True)
